In [1]:
import pandas as pd

df = pd.read_csv('../data/spam_turkce.csv')
print(df.shape)      # kaç satır, kaç sütun
print(df.head(3))
print(df.isnull().sum())  # boş değer var mı?

(5572, 4)
  etiket                                              metin  label  \
0    ham  Go until jurong point, crazy.. Available only ...      0   
1    ham                      Ok lar... Joking wif u oni...      0   
2   spam  Free entry in 2 a wkly comp to win FA Cup fina...      1   

                                            metin_tr  
0  Jurong noktasına kadar git, çılgın.. Sadece bu...  
1              Tamam lar... Seninle şakalaşıyorum...  
2  21 Mayıs 2005 tarihli FA Cup finalini kazanmak...  
etiket      0
metin       0
label       0
metin_tr    2
dtype: int64


In [2]:
# Boş satırları at
df = df.dropna(subset=['metin_tr'])
df = df[df['metin_tr'].str.strip() != '']
print(f"Temiz veri boyutu: {len(df)}")

Temiz veri boyutu: 5570


In [3]:
import re

def metni_temizle(metin):
    metin = str(metin).lower()                        # küçük harfe çevir
    metin = re.sub(r'http\S+|www\S+', '[URL]', metin) # URL'leri etiketle
    metin = re.sub(r'\S+@\S+', '[EMAIL]', metin)      # e-postaları etiketle
    metin = re.sub(r'\d+', '[SAYI]', metin)           # sayıları etiketle
    metin = re.sub(r'[^\w\s]', ' ', metin)            # noktalama sil
    metin = re.sub(r'\s+', ' ', metin).strip()        # fazla boşlukları sil
    return metin

# Test et
ornek = "Tebrikler! 5000 TL kazandınız. Hemen tıklayın: http://odeme.com"
print(metni_temizle(ornek))

tebrikler SAYI tl kazandınız hemen tıklayın URL


In [4]:
# Türkçe anlamsız kelimeler (stopword)
turkce_stopwords = [
    'bir', 've', 'ile', 'bu', 'da', 'de', 'mi', 'mu', 'mı', 'mü',
    'için', 'ama', 'fakat', 'ya', 'ki', 'ne', 'ben', 'sen', 'o',
    'biz', 'siz', 'onlar', 'gibi', 'kadar', 'daha', 'çok', 'en',
    'var', 'yok', 'olan', 'olarak', 'olan', 'diye', 'bile', 'her',
    'hiç', 'hiçbir', 'bazı', 'tüm', 'bütün', 'ise', 'veya', 'hem'
]

def stopword_kaldir(metin):
    kelimeler = metin.split()
    temiz = [k for k in kelimeler if k not in turkce_stopwords]
    return ' '.join(temiz)

# Test
ornek = "bu bir test ve çok önemli bir mesajdır"
print(stopword_kaldir(ornek))

test önemli mesajdır


In [5]:
def tam_on_isleme(metin):
    metin = metni_temizle(metin)
    metin = stopword_kaldir(metin)
    return metin

# Tüm veriye uygula
df['metin_islenmis'] = df['metin_tr'].apply(tam_on_isleme)

# Sonucu gör
print(df[['metin_tr', 'metin_islenmis', 'label']].head(10))

                                            metin_tr  \
0  Jurong noktasına kadar git, çılgın.. Sadece bu...   
1              Tamam lar... Seninle şakalaşıyorum...   
2  21 Mayıs 2005 tarihli FA Cup finalini kazanmak...   
3  Bu kadar erken söylemedin yani... Zaten o zama...   
4  Hayır usf'ye gittiğini sanmıyorum ama buralard...   
5  FreeMsg Hey sevgilim, 3 hafta oldu ve geri dön...   
6  Kardeşim bile benimle konuşmayı sevmiyor. Bana...   
7  Talebiniz doğrultusunda 'Melle Melle (Oru Minn...   
8  KAZANAN!! Değerli bir ağ müşterisi olarak 900 ...   
9  Cep telefonunuz 11 ay veya daha uzun süre mi k...   

                                      metin_islenmis  label  
0  jurong noktasına git çılgın sadece bugis n har...      0  
1                    tamam lar seninle şakalaşıyorum      0  
2  SAYI mayıs SAYI tarihli fa cup finalini kazanm...      1  
3            erken söylemedin yani zaten zaman dedin      0  
4  hayır usf ye gittiğini sanmıyorum buralarda ya...      0  
5  freemsg 

In [6]:
df.to_csv('../data/spam_islenmis.csv', index=False)
print("Kaydedildi! ✓")

# Özet istatistik
print(f"\nToplam kayıt: {len(df)}")
print(f"Spam sayısı: {df['label'].sum()}")
print(f"Gerçek sayısı: {(df['label']==0).sum()}")

Kaydedildi! ✓

Toplam kayıt: 5570
Spam sayısı: 747
Gerçek sayısı: 4823


In [7]:
df = pd.read_csv('../data/spam_islenmis.csv')
print(df[['metin_tr', 'metin_islenmis', 'label']].head(5))
print(df.isnull().sum())

                                            metin_tr  \
0  Jurong noktasına kadar git, çılgın.. Sadece bu...   
1              Tamam lar... Seninle şakalaşıyorum...   
2  21 Mayıs 2005 tarihli FA Cup finalini kazanmak...   
3  Bu kadar erken söylemedin yani... Zaten o zama...   
4  Hayır usf'ye gittiğini sanmıyorum ama buralard...   

                                      metin_islenmis  label  
0  jurong noktasına git çılgın sadece bugis n har...      0  
1                    tamam lar seninle şakalaşıyorum      0  
2  SAYI mayıs SAYI tarihli fa cup finalini kazanm...      1  
3            erken söylemedin yani zaten zaman dedin      0  
4  hayır usf ye gittiğini sanmıyorum buralarda ya...      0  
etiket            0
metin             0
label             0
metin_tr          0
metin_islenmis    1
dtype: int64
